# Day 067 — Solution: Vision Analyzer

In [ ]:
_ANALYZER_SRC = '"""vision_analyzer.py — Day 067: Vision LLM image analyzer.\n\nWraps Ollama llava for image description, classification,\ntext extraction, and multi-task analysis.\n\nSetup:\n    ollama pull llava    # or: ollama pull llama3.2-vision\n\nUsage:\n    from vision_analyzer import VisionAnalyzer\n    from PIL import Image\n\n    va = VisionAnalyzer(model="llava")\n    img = Image.open("photo.jpg")\n    print(va.describe(img))\n    print(va.extract_text(img))\n    print(va.classify(img, ["indoor", "outdoor", "food"]))\n    print(va.analyze(img, tasks=["describe", "objects"]))\n\nTesting without Ollama:\n    mock = lambda b64, prompt: "A red square image."\n    va = VisionAnalyzer(describe_fn=mock)\n"""\nimport io\nimport base64\nfrom PIL import Image\n\n_PROMPTS = {\n    "describe": "Describe this image in 2-3 sentences.",\n    "text":     (\n        "Extract all visible text from this image exactly as it appears. "\n        "If there is no text, reply with an empty string."\n    ),\n    "colors":   "List the 3 dominant colors visible in this image.",\n    "objects":  "List the main objects visible in this image.",\n}\n\n\ndef image_to_base64(img: Image.Image, format: str = "PNG") -> str:\n    """Encode a PIL Image as a base64 string (no data URI prefix)."""\n    buf = io.BytesIO()\n    out = img\n    if format.upper() in ("JPEG", "JPG") and img.mode in ("RGBA", "P"):\n        out = img.convert("RGB")\n    out.save(buf, format=format)\n    return base64.b64encode(buf.getvalue()).decode()\n\n\nclass VisionAnalyzer:\n    """Chainable vision LLM analyzer backed by Ollama.\n\n    Pass describe_fn for testing without a running Ollama server::\n\n        mock = lambda b64, prompt: "A test image."\n        va = VisionAnalyzer(describe_fn=mock)\n    """\n\n    def __init__(self, model: str = "llava",\n                 describe_fn=None) -> None:\n        self.model = model\n        self._describe_fn = describe_fn\n\n    def _call(self, img_b64: str, prompt: str) -> str:\n        if self._describe_fn is not None:\n            return self._describe_fn(img_b64, prompt)\n        import ollama\n        resp = ollama.chat(\n            model=self.model,\n            messages=[{"role": "user", "content": prompt, "images": [img_b64]}],\n        )\n        return resp["message"]["content"]\n\n    def describe(self, img: Image.Image,\n                 prompt: str = "Describe this image in 2-3 sentences.") -> str:\n        """Return a text description of the image."""\n        return self._call(image_to_base64(img), prompt)\n\n    def extract_text(self, img: Image.Image) -> str:\n        """Extract visible text from the image (OCR via vision LLM)."""\n        return self._call(image_to_base64(img), _PROMPTS["text"])\n\n    def classify(self, img: Image.Image, labels: list) -> str:\n        """Zero-shot classify the image into one of the given labels."""\n        label_list = ", ".join(f\'"{l}"\' for l in labels)\n        prompt = (\n            f"Classify this image into exactly one of these categories: "\n            f"{label_list}. Reply with only the category name."\n        )\n        response = self._call(image_to_base64(img), prompt)\n        resp_lower = response.lower()\n        for label in labels:\n            if label.lower() in resp_lower:\n                return label\n        return labels[0]\n\n    def analyze(self, img: Image.Image, tasks: list = None) -> dict:\n        """Run multiple analyses on a single image.\n\n        tasks: list of keys — \'describe\', \'text\', \'colors\', \'objects\'.\n        Returns dict mapping task name to result string.\n        """\n        if tasks is None:\n            tasks = ["describe"]\n        img_b64 = image_to_base64(img)\n        results = {}\n        for task in tasks:\n            if task not in _PROMPTS:\n                raise ValueError(\n                    f"Unknown task: {task!r}. Available: {list(_PROMPTS)}"\n                )\n            results[task] = self._call(img_b64, _PROMPTS[task])\n        return results\n'
from pathlib import Path
Path('vision_analyzer.py').write_text(_ANALYZER_SRC, encoding='utf-8')
print('vision_analyzer.py written.')

In [ ]:
from PIL import Image
import base64, io
from vision_analyzer import VisionAnalyzer, image_to_base64

# Mock: no Ollama required
_mock_fn = lambda b64, prompt: f"[mock] {prompt[:40]}"

va = VisionAnalyzer(describe_fn=_mock_fn)
img = Image.new('RGB', (200, 150), color=(100, 50, 200))

# 1. describe
desc = va.describe(img)
assert isinstance(desc, str) and len(desc) > 0
print("\u2705 describe returns non-empty string")

# 2. describe with custom prompt
custom = va.describe(img, prompt='How many colours?')
assert 'How many colours?' in custom
print("\u2705 custom prompt passed to mock")

# 3. extract_text
text = va.extract_text(img)
assert isinstance(text, str)
print("\u2705 extract_text returns string")

# 4. classify — mock returns the first label's name
labels = ['purple', 'red', 'blue']
cat = va.classify(img, labels)
assert isinstance(cat, str) and cat in labels
print(f"\u2705 classify returns one of the labels: {cat!r}")

# 5. analyze single task
results = va.analyze(img, tasks=['describe'])
assert isinstance(results, dict) and 'describe' in results
print("\u2705 analyze single task returns dict with 'describe' key")

# 6. analyze multi-task
multi = va.analyze(img, tasks=['describe', 'colors'])
assert set(multi.keys()) == {'describe', 'colors'}
print("\u2705 analyze multi-task returns all requested keys")

# 7. analyze unknown task raises ValueError
raised = False
try:
    va.analyze(img, tasks=['teleport'])
except ValueError:
    raised = True
assert raised
print("\u2705 unknown task raises ValueError")

# 8. image_to_base64 produces valid PNG base64
b64 = image_to_base64(img)
raw = base64.b64decode(b64)
assert raw[:4] == b'\x89PNG'
print("\u2705 image_to_base64 produces valid PNG base64")

print("\nVision LLM complete!")
